# SkillForge — Document Roundtrip Tutorial

**Convert between HTML, Markdown, PDF, and AIF using real Wikipedia articles — with honest fidelity checks and fair baselines.**

Three pipelines, each with a lossiness analysis:

1. **HTML → AIF → HTML/MD/LML** — fidelity-checked against cleaned-HTML text
2. **Markdown → AIF → Markdown** — lossless block/inline roundtrip
3. **PDF → AIF → MD/LML** — **text extraction**, not roundtrip (images/fonts/layout dropped)

Every size comparison shows **both bytes and tokens** (GPT-4 `cl100k_base` tokenizer) and includes a fair baseline, not just the adversarial "raw Wikipedia HTML with full chrome."

## Prerequisites

```bash
pip install aif-skillforge tiktoken
cargo build --release -p aif-cli   # needed for Section 3 (PDF)
```

In [1]:
import urllib.request, json, subprocess, re, shutil
from pathlib import Path
from collections import Counter
import skillforge
import tiktoken

WORK = Path('/tmp/aif_roundtrip_demo'); WORK.mkdir(exist_ok=True)
UA = {'User-Agent': 'AIF-Tutorial/1.0'}
ENC = tiktoken.get_encoding('cl100k_base')   # GPT-4 tokenizer (proxy for Claude)

def fetch(url):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=30) as r:
        return r.read()

def kbs(n): return f'{n/1024:.1f} KB' if n < 1_000_000 else f'{n/1024/1024:.2f} MB'
def ntokens(text): return len(ENC.encode(text, disallowed_special=()))

def walk_blocks(blocks):
    """Recursively walk AIF block tree, yielding every block."""
    for b in blocks:
        yield b
        k = b.get('kind', {})
        for child_key in ('children',):
            for child in k.get(child_key, []) or []:
                yield from walk_blocks([child])

def extract_text(blocks):
    """Pull all plain text out of an AIF block tree for fidelity diffing."""
    def inline_text(inlines):
        out = []
        for i in inlines or []:
            t = i.get('type')
            if t == 'Text': out.append(i.get('text', ''))
            elif t in ('Strong', 'Emphasis', 'Code', 'Link'):
                out.append(inline_text(i.get('content', [])))
            elif t == 'SoftBreak': out.append(' ')
        return ''.join(out)
    parts = []
    for b in walk_blocks(blocks):
        k = b.get('kind', {})
        if 'title' in k and k['title']: parts.append(inline_text(k['title']))
        if 'content' in k: parts.append(inline_text(k['content']) if isinstance(k['content'], list) else '')
    return re.sub(r'\s+', ' ', ' '.join(p for p in parts if p)).strip()

print(f'Working dir: {WORK}')
print(f'skillforge: {len([f for f in dir(skillforge) if not f.startswith("_")])} functions')
print(f'Tokenizer: cl100k_base (GPT-4 / close proxy for Claude)')

Working dir: /tmp/aif_roundtrip_demo
skillforge: 17 functions
Tokenizer: cl100k_base (GPT-4 / close proxy for Claude)


## 1 · HTML → AIF → HTML / MD / LML

We fetch **Photosynthesis** as raw Wikipedia HTML, strip chrome, import to AIF IR. Then we:

- render back to HTML, Markdown, and LML Aggressive
- compare sizes in **bytes** and **tokens**
- compare against a **fair text-only baseline** (stripped HTML text)
- verify **text fidelity**: does the roundtrip preserve the article's actual content?

In [2]:
html_raw = fetch('https://en.wikipedia.org/api/rest_v1/page/html/Photosynthesis').decode('utf-8')
(WORK / 'photosynthesis.html').write_text(html_raw)

# Fair baseline: cleaned HTML text (what you'd feed an LLM if you weren't using AIF)
text_baseline = re.sub(r'<[^>]+>', ' ', html_raw)
text_baseline = re.sub(r'\s+', ' ', text_baseline).strip()

print(f'Raw Wikipedia HTML: {kbs(len(html_raw))}, {ntokens(html_raw):,} tokens')
print(f'  (includes nav, infoboxes, citations, inline SVG refs — the adversarial baseline)')
print()
print(f'Cleaned HTML text:  {kbs(len(text_baseline))}, {ntokens(text_baseline):,} tokens')
print(f'  (tags stripped — this is the fair baseline for "what an LLM actually needs")')

Raw Wikipedia HTML: 763.3 KB, 259,203 tokens
  (includes nav, infoboxes, citations, inline SVG refs — the adversarial baseline)

Cleaned HTML text:  109.9 KB, 29,669 tokens
  (tags stripped — this is the fair baseline for "what an LLM actually needs")


In [3]:
# HTML -> AIF IR (strip chrome via readability)
ir_json = skillforge.clean_html(html_raw)
ir = json.loads(ir_json)

# Real block-type histogram: walk children recursively, not just top level
type_counts = Counter()
for b in walk_blocks(ir['blocks']):
    type_counts[b['kind'].get('type', '?')] += 1

print(f'Top-level blocks: {len(ir["blocks"])}')
print(f'All blocks (recursive): {sum(type_counts.values())}')
print()
print('True block distribution (top-level + nested):')
for t, n in type_counts.most_common():
    print(f'  {t:20s} {n}')
print()
print(f'Metadata: {list(ir["metadata"].keys())}')

Top-level blocks: 14
All blocks (recursive): 196

True block distribution (top-level + nested):
  Paragraph            130
  Section              32
  Figure               14
  List                 12
  Table                7
  ThematicBreak        1

Metadata: ['_aif_import_mode', '_aif_source_format', 'title']


In [4]:
html_rt = skillforge.render(ir_json, 'html')
md_rt   = skillforge.render(ir_json, 'markdown')
lml_rt  = skillforge.render(ir_json, 'lml-aggressive')

# Compare bytes + tokens, with both baselines
baselines = {
    'Raw Wikipedia HTML (adversarial)': html_raw,
    'Cleaned HTML text (fair)':          text_baseline,
}
outputs = {
    'AIF HTML (roundtrip)':  html_rt,
    'AIF Markdown':          md_rt,
    'AIF LML Aggressive':    lml_rt,
}

print(f'{"Format":40s} {"Bytes":>10s} {"Tokens":>10s}  {"vs raw HTML":>12s}  {"vs clean text":>14s}')
print('-' * 95)
for name, text in {**baselines, **outputs}.items():
    b, t = len(text), ntokens(text)
    vs_raw  = f'{(1-b/len(html_raw))*100:>11.0f}%' if text is not html_raw else '     —    '
    vs_txt  = f'{(b/len(text_baseline)-1)*100:>+12.0f}%' if text is not text_baseline else '      —     '
    print(f'{name:40s} {b:>10,} {t:>10,}  {vs_raw:>12s}  {vs_txt:>14s}')

print()
print('→ vs raw HTML shows the 75% saving (real, but baseline is chrome-heavy)')
print('→ vs clean text is the honest comparison: AIF costs MORE than plain text,')
print('  but carries typed block structure (sections, figures, links, tables)')

Format                                        Bytes     Tokens   vs raw HTML   vs clean text
-----------------------------------------------------------------------------------------------
Raw Wikipedia HTML (adversarial)            781,575    259,203         —               +594%
Cleaned HTML text (fair)                    112,580     29,669           86%          —     
AIF HTML (roundtrip)                        243,177     77,720           69%           +116%
AIF Markdown                                199,913     62,118           74%            +78%


AIF LML Aggressive                          195,933     58,843           75%            +74%

→ vs raw HTML shows the 75% saving (real, but baseline is chrome-heavy)
→ vs clean text is the honest comparison: AIF costs MORE than plain text,
  but carries typed block structure (sections, figures, links, tables)


In [5]:
# Fidelity check: does roundtrip preserve the actual article text?
# We import once, render to MD, re-import the MD, compare text content.
text_source = extract_text(ir['blocks'])

ir2 = json.loads(skillforge.import_markdown(md_rt))
text_after_md_roundtrip = extract_text(ir2['blocks'])

ir3 = json.loads(skillforge.clean_html(html_rt))
text_after_html_roundtrip = extract_text(ir3['blocks'])

def fidelity(a: str, b: str) -> float:
    from difflib import SequenceMatcher
    return SequenceMatcher(None, a, b).ratio()

print(f'Source text length:              {len(text_source):,} chars')
print(f'After AIF -> MD -> AIF:          {len(text_after_md_roundtrip):,} chars')
print(f'After AIF -> HTML -> AIF:        {len(text_after_html_roundtrip):,} chars')
print()
print(f'Fidelity (text similarity, 1.0 = identical):')
print(f'  MD roundtrip:   {fidelity(text_source, text_after_md_roundtrip):.3f}')
print(f'  HTML roundtrip: {fidelity(text_source, text_after_html_roundtrip):.3f}')
print()
print('Note: text similarity < 1.0 does not mean content was lost — whitespace')
print('normalization and inline formatting can shift chars without losing meaning.')
print('The roundtrip benchmark at benchmarks/roundtrip/ does structural comparison.')

Source text length:              45,404 chars
After AIF -> MD -> AIF:          50,220 chars
After AIF -> HTML -> AIF:        45,418 chars

Fidelity (text similarity, 1.0 = identical):
  MD roundtrip:   0.949


  HTML roundtrip: 1.000

Note: text similarity < 1.0 does not mean content was lost — whitespace
normalization and inline formatting can shift chars without losing meaning.
The roundtrip benchmark at benchmarks/roundtrip/ does structural comparison.


## 2 · Markdown → AIF → Markdown (lossless)

Small hand-written document with headings, lists, tables, quotes, and inline formatting. Roundtrips through AIF IR and back — should be byte-identical at the structure level.

In [6]:
md_source = '''# Photosynthesis

Photosynthesis is a **biological process** used by *plants* to convert light energy into chemical energy.

## Overview

It takes place in the chloroplasts of plant cells.

1. Light-dependent reactions capture sunlight
2. Light-independent reactions (Calvin cycle) fix carbon
3. Products: glucose and oxygen

The chemical equation is `6CO2 + 6H2O + light -> C6H12O6 + 6O2`.

## History

Photosynthesis was first understood by [Jan Ingenhousz](https://en.wikipedia.org/wiki/Jan_Ingenhousz).

> "All plants exhale a fluid that disappears on heating."

| Organism | Type | Habitat |
| --- | --- | --- |
| Spinach | C3 plant | Terrestrial |
| Corn | C4 plant | Terrestrial |
| Cyanobacteria | Prokaryote | Aquatic |
'''
print(f'Source: {len(md_source)} bytes, {ntokens(md_source)} tokens, {len(md_source.splitlines())} lines')

Source: 729 bytes, 191 tokens, 25 lines


In [7]:
# MD -> AIF IR -> MD
ir_json = skillforge.import_markdown(md_source)
ir = json.loads(ir_json)
md_rt = skillforge.render(ir_json, 'markdown')

# Re-import the roundtripped version and verify structural equivalence
ir_rt = json.loads(skillforge.import_markdown(md_rt))

def block_types_tree(blocks):
    return [(b['kind'].get('type'), block_types_tree(b['kind'].get('children', []))) for b in blocks]

print(f'Original MD:    {len(md_source)} bytes, {ntokens(md_source)} tokens')
print(f'Roundtrip MD:   {len(md_rt)} bytes, {ntokens(md_rt)} tokens')
print()
print(f'Block-type tree source:    {block_types_tree(ir["blocks"])}')
print(f'Block-type tree roundtrip: {block_types_tree(ir_rt["blocks"])}')
print()
print(f'Structure preserved: {block_types_tree(ir["blocks"]) == block_types_tree(ir_rt["blocks"])}')
print(f'Text fidelity:       {len(extract_text(ir_rt["blocks"]))/max(1,len(extract_text(ir["blocks"]))):.3f}')
print()
print('--- Roundtripped markdown ---')
print(md_rt)

Original MD:    729 bytes, 191 tokens
Roundtrip MD:   729 bytes, 191 tokens

Block-type tree source:    [('Paragraph', []), ('Section', []), ('Paragraph', []), ('List', []), ('Paragraph', []), ('Section', []), ('Paragraph', []), ('BlockQuote', []), ('Table', [])]
Block-type tree roundtrip: [('Paragraph', []), ('Section', []), ('Paragraph', []), ('List', []), ('Paragraph', []), ('Section', []), ('Paragraph', []), ('BlockQuote', []), ('Table', [])]

Structure preserved: True
Text fidelity:       1.000

--- Roundtripped markdown ---
# Photosynthesis

Photosynthesis is a **biological process** used by *plants* to convert light energy into chemical energy.

## Overview

It takes place in the chloroplasts of plant cells.

1. Light-dependent reactions capture sunlight
2. Light-independent reactions (Calvin cycle) fix carbon
3. Products: glucose and oxygen

The chemical equation is `6CO2 + 6H2O + light -> C6H12O6 + 6O2`.

## History

Photosynthesis was first understood by [Jan Ingenhousz](http

In [8]:
# Same IR -> different formats. Token count reveals the real cost.
print(f'{"Format":20s} {"Bytes":>7s} {"Tokens":>8s}')
print('-' * 38)
for fmt in ['markdown', 'html', 'lml', 'lml-compact', 'lml-aggressive', 'json']:
    out = skillforge.render(ir_json, fmt)
    print(f'{fmt:20s} {len(out):>7d} {ntokens(out):>8d}')
print()
print('→ For LLM ingestion, markdown and lml-aggressive are within a few percent.')
print('→ Bytes and tokens do NOT track linearly: HTML has many 3-char tags that are')
print('  single tokens, so byte-to-token ratio differs across formats.')

Format                 Bytes   Tokens
--------------------------------------
markdown                 729      191
html                    1147      366
lml                      857      225
lml-compact              857      225
lml-aggressive           787      208
json                    6264     1409

→ For LLM ingestion, markdown and lml-aggressive are within a few percent.
→ Bytes and tokens do NOT track linearly: HTML has many 3-char tags that are
  single tokens, so byte-to-token ratio differs across formats.


## 3 · PDF → AIF → MD / LML (text extraction, not roundtrip)

**What this is:** AIF's PDF importer extracts text and classifies blocks as paragraphs, headings, or code. The `avg confidence` score (0.0-1.0) indicates classification certainty.

**What this is NOT:** A true PDF roundtrip. AIF drops images, fonts, layout, figures, headers/footers, and any non-textual content. The re-rendered PDF is text-only with default styling.

**Use case:** Cheap structured text extraction for LLM ingestion or search indexing. For archival or visual preservation, use a proper PDF tool.

PDF support lives in the `aif-pdf` Rust crate (heavy native deps) and is driven via the `aif` CLI binary, not the Python wheel.

In [9]:
AIF = shutil.which('aif')
if AIF is None:
    cur = Path.cwd()
    while cur != cur.parent:
        p = cur / 'target' / 'release' / 'aif'
        if p.exists(): AIF = str(p); break
        cur = cur.parent
assert AIF, 'aif CLI not found — run: cargo build --release -p aif-cli'
print(f'CLI: {AIF}')

CLI: /Users/liqunchen/Claude_Project/token_efficient/target/release/aif


In [10]:
pdf_bytes = fetch('https://en.wikipedia.org/api/rest_v1/page/pdf/Quantum_computing')
src_pdf = WORK / 'quantum.pdf'
src_pdf.write_bytes(pdf_bytes)
print(f'Downloaded PDF: {kbs(len(pdf_bytes))}')

Downloaded PDF: 1.01 MB


In [11]:
# PDF -> AIF JSON (text extraction + block classification)
json_path = WORK / 'quantum.ir.json'
out = subprocess.run([AIF, 'import', str(src_pdf), '-o', str(json_path)],
                     capture_output=True, text=True, check=True)
print('CLI:', (out.stdout + out.stderr).strip())

ir = json.loads(json_path.read_text())
conf = ir['metadata'].get('_aif_import_confidence', '?')
print(f'\nExtracted: {len(ir["blocks"])} blocks, avg confidence {conf}')
print(f'(confidence < 0.7 means many blocks were classified with uncertainty —')
print(f' the PDF importer guesses paragraph vs heading vs code from heuristics)')
print()
kinds = Counter()
for b in ir['blocks']:
    k = b['kind']
    label = k.get('block_type') or k.get('type')
    kinds[label] += 1
print('Block types extracted:')
for t, n in kinds.most_common():
    print(f'  {t:20s} {n}')

CLI: Imported 34 pages, 157 blocks, avg confidence: 0.76
Wrote /tmp/aif_roundtrip_demo/quantum.ir.json

Extracted: 157 blocks, avg confidence 0.76
(confidence < 0.7 means many blocks were classified with uncertainty —
 the PDF importer guesses paragraph vs heading vs code from heuristics)

Block types extracted:
  Paragraph            119
  Section              38


In [12]:
# AIF JSON -> Markdown (the common LLM-ingestion path)
out_md = subprocess.run(
    [AIF, 'compile', '--input-format', 'json', str(json_path), '-f', 'markdown'],
    capture_output=True, text=True, check=True
).stdout
out_lml = subprocess.run(
    [AIF, 'compile', '--input-format', 'json', str(json_path), '-f', 'lml-aggressive'],
    capture_output=True, text=True, check=True
).stdout

print(f'{"Output":30s} {"Bytes":>10s} {"Tokens":>8s}  Note')
print('-' * 80)
print(f'{"Original PDF (with images)":30s} {len(pdf_bytes):>10,} {"—":>8s}  source')
print(f'{"Extracted Markdown":30s} {len(out_md):>10,} {ntokens(out_md):>8,}  text-only')
print(f'{"Extracted LML Aggressive":30s} {len(out_lml):>10,} {ntokens(out_lml):>8,}  text-only')
print()
print('Honest framing:')
print('  - The PDF dropped ~89% of its bytes because images + fonts are gone')
print('  - This is appropriate for LLM ingestion (LLMs can\'t read PDF images anyway)')
print('  - It is NOT appropriate for archival or visual fidelity')
print()
print('--- first 400 chars of extracted markdown ---')
print(out_md[:400])

Output                              Bytes   Tokens  Note
--------------------------------------------------------------------------------
Original PDF (with images)      1,064,020        —  source
Extracted Markdown                115,417   32,706  text-only
Extracted LML Aggressive          115,457   32,749  text-only

Honest framing:
  - The PDF dropped ~89% of its bytes because images + fonts are gone
  - This is appropriate for LLM ingestion (LLMs can't read PDF images anyway)
  - It is NOT appropriate for archival or visual fidelity

--- first 400 chars of extracted markdown ---
# Quantum computing

## Quantum computing

A quantum computer is a (real or theoretical) computer that exploits quantum phenomena like superposition and entanglement in an essential way. It is widely believed that a quantum computer could perform some calculations exponentially faster than any classical computer. For example, a large-scale quantum computer could break some widely used encryption s


In [13]:
# Optional: re-render the extracted text back to PDF (text-only, default fonts)
rt_pdf = WORK / 'quantum.text-only.pdf'
subprocess.run([AIF, 'compile', '--input-format', 'json', str(json_path),
                '-f', 'pdf', '-o', str(rt_pdf)], capture_output=True, check=True)
print(f'Original PDF:       {kbs(src_pdf.stat().st_size)}')
print(f'Text-only PDF:      {kbs(rt_pdf.stat().st_size)}')
print(f'{(1-rt_pdf.stat().st_size/src_pdf.stat().st_size)*100:.0f}% smaller — but that size delta is ENTIRELY images and typography.')
print('The re-rendered PDF has no images, infoboxes, or Wikipedia styling.')

Original PDF:       1.01 MB
Text-only PDF:      123.9 KB
88% smaller — but that size delta is ENTIRELY images and typography.
The re-rendered PDF has no images, infoboxes, or Wikipedia styling.


## Summary — honest framing

| Path | Tool | Loss profile |
|------|------|--------------|
| HTML → AIF → * | Python `clean_html`, `render()` | Drops chrome (nav/footer/ads) + presentational CSS. Preserves content, links, tables, figures. |
| MD → AIF → MD | Python `import_markdown`, `render()` | Lossless at the block-structure level. Inline formatting preserved. |
| PDF → AIF → text | CLI `aif import`/`compile` | **Lossy by design.** Drops images, fonts, layout. Extracts text + heading/code classification with a confidence score. |

### When AIF's token savings matter

- **vs. raw Wikipedia HTML:** 75% smaller — but raw HTML is the adversarial baseline (full chrome)
- **vs. cleaned HTML text:** AIF costs **more** tokens, because it carries typed block structure (sections, tables, figures, links)
- **vs. Markdown:** roughly the same token count — AIF is not a token-efficiency play against MD

### When to use each output format

| If you need… | Use |
|---|---|
| LLM context (general) | `markdown` — universal, human-readable, ~same tokens as LML |
| LLM context (typed skills) | `lml-aggressive` — typed `@step:`/`@verify:` blocks, ~same tokens |
| Browser rendering | `html` |
| Storage / transport | `json` or `binary-*` |
| PDF archival | **not AIF** — use a proper PDF library |

All generated files are in `/tmp/aif_roundtrip_demo/` — inspect the .md and .lml files to see structural extraction quality on a real Wikipedia article.